# Zero-shot-SLD — Zero-shot Spatial Layout for Diffusion

## Ideia

O **Zero-shot-SLD** realiza o planejamento espacial **antes da geração da imagem**.

### Modo 1 — somente prompt

**Prompt → LLM → objetos + bounding boxes → GLIGEN → imagem final**

A LLM interpreta o prompt, identifica quais entidades concretas precisam aparecer e
cria uma composição espacial plausível em zero-shot. As bounding boxes condicionam
a geração desde o primeiro passo.

### Modo 2 — prompt + rascunho opcional

**Prompt + rascunho → LLM multimodal → composição unificada → objetos + bounding boxes → GLIGEN → imagem final**

O rascunho é usado **somente como guia de composição**. Ele nunca é enviado ao
modelo de difusão.

A LLM deve construir a união semântica entre prompt e rascunho:

- objeto presente no prompt e visível no rascunho → preserva sua posição aproximada;
- objeto citado no prompt, mas ausente do rascunho → cria uma nova bounding box em posição plausível;
- objeto visualmente relevante no rascunho, mas ausente do prompt → adiciona esse objeto ao layout;
- conceitos abstratos (estilo, iluminação, atmosfera etc.) não recebem bounding boxes.

A saída da LLM contém:

1. `generation_prompt`: prompt final consolidado;
2. `objects`: objetos concretos;
3. `box`: bounding box de cada objeto.

O gerador recebe **somente**:

- `generation_prompt`;
- frases dos objetos;
- bounding boxes.

O rascunho não participa da difusão.


In [ ]:
# ============================================================
# 1. Instalação reproduzível das dependências
# ============================================================

!pip install -q \
    "huggingface_hub==0.27.1" \
    "transformers==4.48.3" \
    "accelerate==1.3.0" \
    "diffusers==0.32.2" \
    "google-genai>=1.0.0" \
    "pydantic>=2.0" \
    "Pillow>=10.0" \
    "pandas>=2.0" \
    "gdown>=5.0"

In [ ]:
# ============================================================
# 2. Imports, GPU e configuração geral
# ============================================================
import os
import json
import torch
import pandas as pd

from PIL import Image, ImageDraw
from IPython.display import display
from pydantic import BaseModel, Field
from typing import List

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU CUDA não encontrada. No Colab, vá em "
        "Ambiente de execução > Alterar tipo de ambiente de execução > GPU."
    )

DEVICE = "cuda"
DTYPE = torch.float16
IMAGE_SIZE = 512
SEED = 42

print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# 3. Dataset: rascunhos + prompts
# ============================================================
import os
import gdown

DRAFTS_FOLDER_ID = "1DQNw83XSfKKS4XjfZFo6V4Fz05swxJTk"

PROMPTS_SHEET_URL = (
    "https://docs.google.com/spreadsheets/d/"
    "1GvyTi1A9yo4lE8_88Sng1IlDi3YEDFK7dEamXohHq2A/edit#gid=0"
)

DRAFTS_DIR = "/content/drafts"
OUTPUT_DIR = "/content/resultados_mgi"

os.makedirs(DRAFTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Baixa a pasta apenas se ainda não houver arquivos.
has_files = any(
    os.path.isfile(os.path.join(root, name))
    for root, _, files in os.walk(DRAFTS_DIR)
    for name in files
)

if not has_files:
    print("Baixando rascunhos...")
    gdown.download_folder(
        id=DRAFTS_FOLDER_ID,
        output=DRAFTS_DIR,
        quiet=False,
        use_cookies=False,
    )
else:
    print("Rascunhos já encontrados. Download ignorado.")

# Carrega a planilha.
sheet_id = PROMPTS_SHEET_URL.split("/d/")[1].split("/")[0]
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid=0"

df_prompts = pd.read_csv(csv_url)

print(f"Dataset carregado: {len(df_prompts)} amostras")
print("Colunas:", list(df_prompts.columns))
display(df_prompts.head())

# Diagnóstico: mostra os primeiros arquivos encontrados recursivamente.
downloaded_files = []
for root, _, files in os.walk(DRAFTS_DIR):
    for name in files:
        downloaded_files.append(os.path.join(root, name))

print(f"\nArquivos de rascunho encontrados: {len(downloaded_files)}")
for path in downloaded_files[:20]:
    print(" -", os.path.relpath(path, DRAFTS_DIR))

if not downloaded_files:
    raise RuntimeError(
        "Nenhuma imagem foi encontrada em /content/drafts após o download."
    )


In [ ]:
# ============================================================
# 4. Zero-shot Layout Planner
#    prompt (+ rascunho opcional) -> layout semântico
# ============================================================

from google import genai
from google.colab import userdata
from pydantic import BaseModel, Field
from typing import List

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise RuntimeError(
        "Secret GEMINI_API_KEY não encontrado. "
        "Adicione a chave em Colab > Secrets."
    )

client = genai.Client(api_key=GEMINI_API_KEY)

# Pode ser substituído por outro modelo multimodal compatível.
GEMINI_MODEL = "gemini-3.5-flash-lite"


class LayoutObject(BaseModel):
    phrase: str = Field(
        description="Short English noun phrase for one concrete visual object."
    )
    box: List[float] = Field(
        description=(
            "Bounding box in [ymin, xmin, ymax, xmax] format "
            "using Gemini's 0..1000 coordinate convention."
        ),
        min_length=4,
        max_length=4,
    )


class LayoutPlan(BaseModel):
    generation_prompt: str = Field(
        description=(
            "Final English generation prompt containing the union of relevant "
            "content from the user prompt and optional sketch."
        )
    )
    objects: List[LayoutObject]


def gemini_box_to_gligen(box):
    """
    Gemini:
        [ymin, xmin, ymax, xmax], normalmente 0..1000

    GLIGEN:
        [xmin, ymin, xmax, ymax], normalizado 0..1
    """
    if box is None or len(box) != 4:
        return None

    try:
        ymin, xmin, ymax, xmax = [float(v) for v in box]
    except (TypeError, ValueError):
        return None

    scale = (
        1000.0
        if max(abs(v) for v in [ymin, xmin, ymax, xmax]) > 1.0
        else 1.0
    )

    ymin /= scale
    xmin /= scale
    ymax /= scale
    xmax /= scale

    xmin = max(0.0, min(1.0, xmin))
    ymin = max(0.0, min(1.0, ymin))
    xmax = max(0.0, min(1.0, xmax))
    ymax = max(0.0, min(1.0, ymax))

    if xmax <= xmin or ymax <= ymin:
        return None

    # Evita caixas degeneradas.
    if (xmax - xmin) < 0.01 or (ymax - ymin) < 0.01:
        return None

    return [xmin, ymin, xmax, ymax]


def build_zero_shot_layout(user_prompt, sketch_image=None):
    """
    Cria o plano espacial ANTES da geração.

    sketch_image:
        None  -> planejamento zero-shot somente a partir do prompt.
        PIL   -> o rascunho é usado pela LLM como guia espacial, mas NÃO
                 será enviado ao GLIGEN.
    """

    has_sketch = sketch_image is not None

    if has_sketch:
        task_description = """
You also receive a rough user sketch.

Treat the sketch ONLY as a semantic and spatial guide.

Build the UNION of the text prompt and the sketch:

1. If an object is requested in the prompt and appears in the sketch,
   use the sketch to estimate its position.
2. If an object is requested by the prompt but missing from the sketch,
   ADD it to the layout in a plausible free region.
3. If an important concrete object is clearly present in the sketch but
   not mentioned in the prompt, ADD it to the layout and include it in
   generation_prompt.
4. Do NOT try to reproduce pixels, colors, texture, line style, or visual
   imperfections from the sketch.
5. The sketch itself will NOT be passed to the image generator.
"""
    else:
        task_description = """
There is no sketch.

Create a plausible image composition entirely in zero-shot from the text.
Choose positions and sizes that make the requested scene visually coherent.
Every important concrete entity explicitly required by the prompt should
receive a bounding box when spatial conditioning is useful.
"""

    instruction = f"""
You are the ZERO-SHOT SPATIAL PLANNER of a text-to-image system.

USER PROMPT:
"{user_prompt}"

{task_description}

Your job happens BEFORE image generation.

Produce:
- generation_prompt: one concise but complete ENGLISH image-generation prompt;
- objects: the concrete entities that should receive spatial grounding.

Rules:
- Preserve all concrete requirements from the user's text.
- Do not omit requested objects because they are absent from the sketch.
- Do not create bounding boxes for abstract style concepts, lighting,
  realism, mood, quality adjectives, actions, or relationships themselves.
- Relationships such as "dog playing with frisbee" should instead be
  expressed through the positions of "dog" and "frisbee".
- For counts, return separate boxes when separate instances are requested,
  e.g. "three dogs" -> three dog entries when feasible.
- Use short English phrases suitable for GLIGEN.
- Bounding box format: [ymin, xmin, ymax, xmax].
- Coordinates: integers/floats from 0 to 1000.
- y increases top-to-bottom; x increases left-to-right.
- Avoid duplicate boxes for the same physical instance.
- Keep boxes inside the canvas.
- Make the composition spatially plausible.
"""

    contents = [instruction]

    if has_sketch:
        contents.append(sketch_image)

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=contents,
        config={
            "response_mime_type": "application/json",
            "response_schema": LayoutPlan,
            "temperature": 0.15,
        },
    )

    if getattr(response, "parsed", None) is not None:
        parsed = response.parsed
        if isinstance(parsed, LayoutPlan):
            plan = parsed
        else:
            plan = LayoutPlan.model_validate(parsed)
    else:
        plan = LayoutPlan.model_validate_json(response.text)

    phrases = []
    boxes = []

    print("Plano bruto da LLM:")
    print("generation_prompt:", plan.generation_prompt)

    for obj in plan.objects:
        print("  ", obj.phrase, obj.box)

        converted = gemini_box_to_gligen(obj.box)
        phrase = obj.phrase.strip()

        if phrase and converted is not None:
            phrases.append(phrase)
            boxes.append(converted)

    if not boxes:
        raise ValueError(
            "A LLM não produziu nenhuma bounding box válida para o layout."
        )

    print("\nPlano convertido para GLIGEN:")
    for phrase, box in zip(phrases, boxes):
        print("  ", phrase, box)

    return {
        "generation_prompt": plan.generation_prompt.strip(),
        "phrases": phrases,
        "boxes": boxes,
    }


def draw_layout(image, phrases, boxes):
    """
    Visualiza o plano GLIGEN sobre uma imagem-base.
    A imagem-base pode ser o rascunho ou um canvas branco.
    """
    vis = image.copy().convert("RGB")
    draw = ImageDraw.Draw(vis)

    W, H = vis.size

    for i, (phrase, box) in enumerate(zip(phrases, boxes)):
        xmin, ymin, xmax, ymax = box

        x1 = int(xmin * W)
        y1 = int(ymin * H)
        x2 = int(xmax * W)
        y2 = int(ymax * H)

        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)

        label = f"{i}: {phrase}"
        text_y = max(0, y1 - 18)
        label_w = min(W - x1, max(80, len(label) * 7))

        draw.rectangle(
            [x1, text_y, x1 + label_w, y1],
            fill="red",
        )
        draw.text(
            (x1 + 2, text_y + 2),
            label,
            fill="white",
        )

    return vis


In [ ]:
# ============================================================
# 5. GLIGEN — geração condicionada por texto + bounding boxes
# ============================================================

from diffusers import StableDiffusionGLIGENPipeline

# IMPORTANTE:
# Este checkpoint possui o UNet configurado com:
#     attention_type = "gated"
#
# Isso é necessário para que o Diffusers crie:
#     unet.position_net
#
# que transforma as bounding boxes + embeddings de texto
# em grounding para o GLIGEN.
GLIGEN_MODEL_ID = "masterful/gligen-1-4-generation-text-box"

print("Carregando:", GLIGEN_MODEL_ID)

pipe = StableDiffusionGLIGENPipeline.from_pretrained(
    GLIGEN_MODEL_ID,
    variant="fp16",
    torch_dtype=torch.float16,
)

pipe = pipe.to(DEVICE)

# Reduz pico de VRAM sem mover módulos para CPU.
pipe.enable_attention_slicing()

print("GLIGEN carregado com sucesso.")
print("Device do VAE:", next(pipe.vae.parameters()).device)
print("Device do UNet:", next(pipe.unet.parameters()).device)
print("UNet in_channels:", pipe.unet.config.in_channels)
print(
    "UNet attention_type:",
    getattr(pipe.unet.config, "attention_type", None),
)
print(
    "UNet possui position_net:",
    hasattr(pipe.unet, "position_net"),
)

# ------------------------------------------------------------
# Validações importantes
# ------------------------------------------------------------

if pipe.unet.config.in_channels != 4:
    raise RuntimeError(
        f"Era esperado um UNet de geração com 4 canais, "
        f"mas foram encontrados {pipe.unet.config.in_channels}."
    )

attention_type = getattr(
    pipe.unet.config,
    "attention_type",
    None,
)

if attention_type not in ("gated", "gated-text-image"):
    raise RuntimeError(
        "O checkpoint carregado não possui atenção GLIGEN habilitada. "
        f"attention_type={attention_type!r}. "
        "O Zero-shot-SLD precisa de attention_type='gated'."
    )

if not hasattr(pipe.unet, "position_net"):
    raise RuntimeError(
        "O UNet foi carregado sem 'position_net'. "
        "Isso significa que o grounding espacial do GLIGEN não foi "
        "inicializado corretamente."
    )

print("\n✓ Checkpoint GLIGEN válido.")
print("✓ position_net presente.")
print("✓ Gated attention presente.")
print("✓ Pronto para receber bounding boxes.")


In [ ]:
# ============================================================
# 6. Geração Zero-shot-SLD
# ============================================================

def generate_from_layout(
    generation_prompt,
    phrases,
    boxes,
    seed=42,
    num_inference_steps=35,
    guidance_scale=7.5,
    scheduled_sampling_beta=0.4,
):
    """
    A imagem final é gerada do zero.

    Entradas:
      - generation_prompt
      - frases dos objetos
      - bounding boxes

    Nenhum pixel do rascunho é enviado ao GLIGEN.
    """

    if len(phrases) != len(boxes):
        raise ValueError(
            f"phrases={len(phrases)} e boxes={len(boxes)}."
        )

    if not phrases:
        raise ValueError(
            "Nenhum objeto foi fornecido para o grounding espacial."
        )

    if not hasattr(pipe.unet, "position_net"):
        raise RuntimeError(
            "UNet sem position_net. "
            "Execute novamente a célula 5 com o checkpoint GLIGEN correto."
        )

    generator = torch.Generator(
        device=DEVICE
    ).manual_seed(seed)

    print(
        "Devices antes da inferência:",
        "VAE =", next(pipe.vae.parameters()).device,
        "| UNet =", next(pipe.unet.parameters()).device,
        "| channels =", pipe.unet.config.in_channels,
        "| attention_type =",
        getattr(pipe.unet.config, "attention_type", None),
        "| position_net =",
        hasattr(pipe.unet, "position_net"),
    )

    print("\nGrounding enviado ao GLIGEN:")
    for phrase, box in zip(phrases, boxes):
        print("  ", phrase, "->", box)

    with torch.inference_mode():
        result = pipe(
            prompt=generation_prompt,
            gligen_phrases=phrases,
            gligen_boxes=boxes,
            gligen_scheduled_sampling_beta=scheduled_sampling_beta,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            generator=generator,
            height=IMAGE_SIZE,
            width=IMAGE_SIZE,
            output_type="pil",
        )

    return result.images[0]


In [ ]:
# ============================================================
# 6.1. Função auxiliar para localizar os rascunhos
# ============================================================
from pathlib import Path

IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp", ".bmp"]


def find_draft_file(draft_value, drafts_dir=DRAFTS_DIR):
    """
    Localiza a imagem indicada pela coluna 'draft'.

    Aceita:
    - nome exato: exemplo.png
    - caminho relativo: pasta/exemplo.png
    - nome sem extensão: exemplo
    - diferenças simples de maiúsculas/minúsculas
    - arquivos dentro de subpastas criadas pelo gdown
    """
    if pd.isna(draft_value):
        return None

    raw = str(draft_value).strip()
    if not raw:
        return None

    base_dir = Path(drafts_dir)

    # 1) Caminho direto relativo ao diretório de rascunhos.
    direct = base_dir / raw
    if direct.is_file():
        return str(direct)

    # Remove eventual prefixo de pasta e trabalha também com o basename.
    raw_name = Path(raw).name
    raw_stem = Path(raw_name).stem
    raw_suffix = Path(raw_name).suffix.lower()

    # 2) Procura recursivamente pelo nome exato.
    for path in base_dir.rglob("*"):
        if path.is_file() and path.name.lower() == raw_name.lower():
            return str(path)

    # 3) Se veio sem extensão, tenta extensões comuns.
    if not raw_suffix:
        for ext in IMAGE_EXTENSIONS:
            target_name = (raw_name + ext).lower()
            for path in base_dir.rglob("*"):
                if path.is_file() and path.name.lower() == target_name:
                    return str(path)

    # 4) Última tentativa: compara apenas o stem.
    matches = []
    for path in base_dir.rglob("*"):
        if (
            path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
            and path.stem.lower() == raw_stem.lower()
        ):
            matches.append(path)

    if len(matches) == 1:
        return str(matches[0])

    if len(matches) > 1:
        print(
            f"Aviso: múltiplos arquivos correspondem a '{raw}'. "
            f"Usando: {matches[0]}"
        )
        return str(matches[0])

    return None


In [ ]:
# ============================================================
# 7. Teste em UMA amostra
# ============================================================

TEST_INDEX = 0

# True:
#   prompt + rascunho -> LLM planeja usando ambos
#
# False:
#   somente prompt -> LLM cria todo o layout em zero-shot
USE_DRAFT_GUIDANCE = True

row = df_prompts.iloc[TEST_INDEX]

user_prompt = str(row["prompt"]).strip()

draft_img = None
draft_path = None
draft_value = None

if USE_DRAFT_GUIDANCE:
    if "draft" not in df_prompts.columns or pd.isna(row["draft"]):
        print("Nenhum rascunho disponível; usando modo prompt-only.")
    else:
        draft_value = str(row["draft"]).strip()
        draft_path = find_draft_file(draft_value)

        if draft_path is None:
            print(
                f"Rascunho '{draft_value}' não encontrado. "
                "Usando modo prompt-only."
            )
        else:
            draft_img = (
                Image.open(draft_path)
                .convert("RGB")
                .resize((IMAGE_SIZE, IMAGE_SIZE))
            )

print("ID:", row["id"] if "id" in df_prompts.columns else TEST_INDEX)
print("Prompt original:", user_prompt)
print("Modo:", "PROMPT + RASCUNHO" if draft_img is not None else "PROMPT ONLY")

if draft_img is not None:
    print("Rascunho usado SOMENTE pela LLM:", draft_path)
    display(draft_img)

# ------------------------------------------------------------
# 1. Planejamento espacial antes da geração
# ------------------------------------------------------------
layout = build_zero_shot_layout(
    user_prompt=user_prompt,
    sketch_image=draft_img,
)

generation_prompt = layout["generation_prompt"]
phrases = layout["phrases"]
boxes = layout["boxes"]

print("\nPrompt final de geração:")
print(generation_prompt)

print("\nObjetos planejados:")
for phrase, box in zip(phrases, boxes):
    print("-", phrase, box)

# ------------------------------------------------------------
# 2. Visualização do plano
# ------------------------------------------------------------
if draft_img is not None:
    # Apenas para visualizar se as caixas seguem o rascunho.
    # Esta imagem NÃO é enviada ao GLIGEN.
    layout_canvas = draft_img
else:
    layout_canvas = Image.new(
        "RGB",
        (IMAGE_SIZE, IMAGE_SIZE),
        "white",
    )

layout_vis = draw_layout(
    layout_canvas,
    phrases,
    boxes,
)
display(layout_vis)

# ------------------------------------------------------------
# 3. Geração do zero usando SOMENTE texto + caixas
# ------------------------------------------------------------
final_image = generate_from_layout(
    generation_prompt=generation_prompt,
    phrases=phrases,
    boxes=boxes,
    seed=SEED,
)

display(final_image)

test_output = os.path.join(
    OUTPUT_DIR,
    "zero_shot_sld_test.png",
)
final_image.save(test_output)

print("Resultado salvo em:", test_output)


In [ ]:
# ============================================================
# 8. Execução no dataset completo
# ============================================================

# Altere para False para avaliar o Zero-shot-SLD apenas com texto.
USE_DRAFT_GUIDANCE = True

results_log = []

for index, row in df_prompts.iterrows():
    try:
        user_prompt = str(row["prompt"]).strip()

        if not user_prompt:
            print(f"[{index}] Prompt vazio. Pulando.")
            continue

        draft_img = None
        draft_path = None
        draft_value = None

        if (
            USE_DRAFT_GUIDANCE
            and "draft" in df_prompts.columns
            and pd.notna(row["draft"])
        ):
            draft_value = str(row["draft"]).strip()
            draft_path = find_draft_file(draft_value)

            if draft_path is not None:
                draft_img = (
                    Image.open(draft_path)
                    .convert("RGB")
                    .resize((IMAGE_SIZE, IMAGE_SIZE))
                )

        print("\n" + "=" * 70)
        print(f"[{index + 1}/{len(df_prompts)}]")
        print("Prompt:", user_prompt)
        print(
            "Modo:",
            "prompt+sketch" if draft_img is not None else "prompt-only",
        )

        # 1. Planejamento pré-geração
        layout = build_zero_shot_layout(
            user_prompt=user_prompt,
            sketch_image=draft_img,
        )

        generation_prompt = layout["generation_prompt"]
        phrases = layout["phrases"]
        boxes = layout["boxes"]

        # 2. Geração do zero
        final_image = generate_from_layout(
            generation_prompt=generation_prompt,
            phrases=phrases,
            boxes=boxes,
            seed=SEED,
        )

        sample_id = (
            str(row["id"])
            if "id" in df_prompts.columns
            else str(index)
        )

        # 3. Visualização do plano
        if draft_img is not None:
            canvas = draft_img
        else:
            canvas = Image.new(
                "RGB",
                (IMAGE_SIZE, IMAGE_SIZE),
                "white",
            )

        layout_vis = draw_layout(
            canvas,
            phrases,
            boxes,
        )

        layout_path = os.path.join(
            OUTPUT_DIR,
            f"{sample_id}_layout.jpg",
        )
        generated_path = os.path.join(
            OUTPUT_DIR,
            f"{sample_id}_generated.png",
        )
        json_path = os.path.join(
            OUTPUT_DIR,
            f"{sample_id}_plan.json",
        )

        layout_vis.save(layout_path)
        final_image.save(generated_path)

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "id": sample_id,
                    "mode": (
                        "prompt+sketch"
                        if draft_img is not None
                        else "prompt-only"
                    ),
                    "user_prompt": user_prompt,
                    "generation_prompt": generation_prompt,
                    "draft": draft_value,
                    "phrases": phrases,
                    "boxes": boxes,
                },
                f,
                ensure_ascii=False,
                indent=2,
            )

        results_log.append({
            "id": sample_id,
            "status": "ok",
            "mode": (
                "prompt+sketch"
                if draft_img is not None
                else "prompt-only"
            ),
            "num_boxes": len(boxes),
            "generated_path": generated_path,
        })

        print("Resultado:", generated_path)

        del final_image
        torch.cuda.empty_cache()

    except Exception as e:
        print(
            f"ERRO [{index}]: "
            f"{type(e).__name__}: {e}"
        )

        results_log.append({
            "id": (
                str(row["id"])
                if "id" in df_prompts.columns
                else str(index)
            ),
            "status": "error",
            "error": str(e),
        })

results_df = pd.DataFrame(results_log)
display(results_df)

log_path = os.path.join(
    OUTPUT_DIR,
    "zero_shot_sld_results.csv",
)
results_df.to_csv(log_path, index=False)

print("Log salvo em:", log_path)


In [ ]:
# ============================================================
# 9. Visualização de TODO o dataset em uma única imagem
# ============================================================

import os
import json
import math
import textwrap
import matplotlib.pyplot as plt
from PIL import Image

RESULTS_DIR = "/content/resultados_mgi"
DRAFTS_DIR = "/content/drafts"

# ------------------------------------------------------------
# Encontra todas as imagens geradas
# ------------------------------------------------------------
generated_files = sorted([
    f for f in os.listdir(RESULTS_DIR)
    if f.endswith("_generated.png")
])

if not generated_files:
    raise RuntimeError(
        f"Nenhum arquivo *_generated.png encontrado em {RESULTS_DIR}"
    )

print(f"Amostras encontradas: {len(generated_files)}")

# ------------------------------------------------------------
# Quantidade de linhas e colunas
#
# Cada linha:
#   Rascunho | Bounding Boxes | Resultado
# ------------------------------------------------------------
num_samples = len(generated_files)
num_cols = 3

# Altura por amostra.
# Se houver muitas imagens, pode diminuir de 5 para 4.
row_height = 5

fig, axes = plt.subplots(
    num_samples,
    num_cols,
    figsize=(15, row_height * num_samples),
)

# Caso exista somente uma amostra
if num_samples == 1:
    axes = [axes]

# ------------------------------------------------------------
# Processa cada resultado
# ------------------------------------------------------------
for row_idx, generated_file in enumerate(generated_files):

    sample_id = generated_file.replace(
        "_generated.png",
        ""
    )

    generated_path = os.path.join(
        RESULTS_DIR,
        generated_file
    )

    layout_path = os.path.join(
        RESULTS_DIR,
        f"{sample_id}_layout.jpg"
    )

    plan_path = os.path.join(
        RESULTS_DIR,
        f"{sample_id}_plan.json"
    )

    # --------------------------------------------------------
    # Carrega JSON da amostra
    # --------------------------------------------------------
    plan = {}

    if os.path.exists(plan_path):
        with open(
            plan_path,
            "r",
            encoding="utf-8"
        ) as f:
            plan = json.load(f)

    user_prompt = plan.get(
        "user_prompt",
        "Prompt não encontrado"
    )

    generation_prompt = plan.get(
        "generation_prompt",
        ""
    )

    draft_name = plan.get(
        "draft",
        None
    )

    # --------------------------------------------------------
    # Carrega rascunho
    # --------------------------------------------------------
    draft_image = None

    if draft_name:

        if "find_draft_file" in globals():
            draft_path = find_draft_file(
                draft_name
            )
        else:
            draft_path = os.path.join(
                DRAFTS_DIR,
                draft_name
            )

        if (
            draft_path is not None
            and os.path.exists(draft_path)
        ):
            draft_image = Image.open(
                draft_path
            ).convert("RGB")

    # Caso não exista rascunho
    if draft_image is None:
        draft_image = Image.new(
            "RGB",
            (512, 512),
            "white"
        )

    # --------------------------------------------------------
    # Bounding boxes
    # --------------------------------------------------------
    if os.path.exists(layout_path):
        layout_image = Image.open(
            layout_path
        ).convert("RGB")
    else:
        layout_image = Image.new(
            "RGB",
            (512, 512),
            "white"
        )

    # --------------------------------------------------------
    # Imagem final
    # --------------------------------------------------------
    final_image = Image.open(
        generated_path
    ).convert("RGB")

    # --------------------------------------------------------
    # Mostra as 3 imagens
    # --------------------------------------------------------
    ax_draft = axes[row_idx][0]
    ax_layout = axes[row_idx][1]
    ax_final = axes[row_idx][2]

    ax_draft.imshow(draft_image)
    ax_layout.imshow(layout_image)
    ax_final.imshow(final_image)

    ax_draft.axis("off")
    ax_layout.axis("off")
    ax_final.axis("off")

    # --------------------------------------------------------
    # Títulos
    # --------------------------------------------------------
    if row_idx == 0:
        ax_draft.set_title(
            "Rascunho",
            fontsize=16,
            fontweight="bold"
        )

        ax_layout.set_title(
            "Bounding Boxes",
            fontsize=16,
            fontweight="bold"
        )

        ax_final.set_title(
            "Resultado Zero-shot-SLD",
            fontsize=16,
            fontweight="bold"
        )

    # --------------------------------------------------------
    # Prompt abaixo da primeira imagem da linha
    # --------------------------------------------------------
    wrapped_prompt = "\n".join(
        textwrap.wrap(
            user_prompt,
            width=60
        )
    )

    ax_draft.text(
        0.5,
        -0.08,
        f"ID: {sample_id}\n{wrapped_prompt}",
        transform=ax_draft.transAxes,
        ha="center",
        va="top",
        fontsize=10
    )


# ------------------------------------------------------------
# Título geral
# ------------------------------------------------------------
fig.suptitle(
    "Zero-shot-SLD — Resultados do Dataset",
    fontsize=22,
    fontweight="bold",
    y=0.995
)

plt.tight_layout(
    rect=[0, 0, 1, 0.985],
    h_pad=4
)

# ------------------------------------------------------------
# Salva uma única imagem
# ------------------------------------------------------------
OUTPUT_PATH = os.path.join(
    RESULTS_DIR,
    "zero_shot_sld_dataset_completo.png"
)

plt.savefig(
    OUTPUT_PATH,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("\nImagem completa salva em:")
print(OUTPUT_PATH)

In [ ]:
# ============================================================
# 10. Comparação entre os quatro métodos
# ============================================================
#
# 1. Prompt only
# 2. Img2Img (strength = 0.7)
# 3. Zero-shot-SLD: Prompt -> LLM -> Bounding Boxes -> GLIGEN
# 4. Zero-shot-SLD: Prompt + Sketch -> LLM -> Bounding Boxes -> GLIGEN
#
# IMPORTANTE:
# O método 4 NÃO envia o rascunho para o GLIGEN.
# O rascunho é usado somente pela LLM para planejar as caixas.
# ============================================================

import os
import gc
import torch
import matplotlib.pyplot as plt
import textwrap

from PIL import Image
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionImg2ImgPipeline
)

# ------------------------------------------------------------
# CONFIGURAÇÕES DA COMPARAÇÃO
# ------------------------------------------------------------

COMPARISON_INDEX = 0

COMPARISON_SEED = 42
COMPARISON_STEPS = 35
COMPARISON_GUIDANCE = 7.5

IMG2IMG_STRENGTH = 0.7

BASE_MODEL_ID = "CompVis/stable-diffusion-v1-4"

COMPARISON_DIR = os.path.join(
    OUTPUT_DIR,
    "comparacao"
)

os.makedirs(
    COMPARISON_DIR,
    exist_ok=True
)


# ============================================================
# A. CARREGA A AMOSTRA
# ============================================================

comparison_row = df_prompts.iloc[
    COMPARISON_INDEX
]

comparison_id = (
    str(comparison_row["id"])
    if "id" in df_prompts.columns
    else str(COMPARISON_INDEX)
)

comparison_prompt = str(
    comparison_row["prompt"]
).strip()

comparison_draft = None
comparison_draft_path = None

if (
    "draft" in df_prompts.columns
    and pd.notna(comparison_row["draft"])
):

    comparison_draft_name = str(
        comparison_row["draft"]
    ).strip()

    comparison_draft_path = find_draft_file(
        comparison_draft_name
    )

    if comparison_draft_path:
        comparison_draft = (
            Image.open(comparison_draft_path)
            .convert("RGB")
            .resize(
                (IMAGE_SIZE, IMAGE_SIZE)
            )
        )


if comparison_draft is None:
    raise RuntimeError(
        "Esta comparação precisa de uma imagem de referência "
        "para executar o método Img2Img e o método multimodal."
    )


print("=" * 70)
print("COMPARAÇÃO ZERO-SHOT-SLD")
print("=" * 70)

print("ID:", comparison_id)
print("Prompt:", comparison_prompt)
print(
    "Rascunho:",
    comparison_draft_path
)

display(comparison_draft)


# ============================================================
# B. MÉTODO 1 — SOMENTE PROMPT
# ============================================================

print("\n" + "=" * 70)
print("1/4 — TEXT-TO-IMAGE — SOMENTE PROMPT")
print("=" * 70)


# ------------------------------------------------------------
# Temporariamente libera GLIGEN da GPU
# para evitar estouro de VRAM
# ------------------------------------------------------------

pipe.to("cpu")
torch.cuda.empty_cache()
gc.collect()


txt2img_pipe = (
    StableDiffusionPipeline
    .from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16,
        safety_checker=None,
    )
    .to(DEVICE)
)

txt2img_pipe.enable_attention_slicing()


generator = torch.Generator(
    device=DEVICE
).manual_seed(
    COMPARISON_SEED
)


with torch.inference_mode():

    image_prompt_only = txt2img_pipe(

        prompt=comparison_prompt,

        num_inference_steps=(
            COMPARISON_STEPS
        ),

        guidance_scale=(
            COMPARISON_GUIDANCE
        ),

        generator=generator,

        height=IMAGE_SIZE,
        width=IMAGE_SIZE,

    ).images[0]


prompt_only_path = os.path.join(
    COMPARISON_DIR,
    f"{comparison_id}_01_prompt_only.png"
)

image_prompt_only.save(
    prompt_only_path
)

print(
    "Salvo:",
    prompt_only_path
)


# ============================================================
# C. MÉTODO 2 — IMG2IMG
# ============================================================

print("\n" + "=" * 70)
print(
    "2/4 — IMG2IMG — "
    f"strength={IMG2IMG_STRENGTH}"
)
print("=" * 70)


# ------------------------------------------------------------
# Reaproveita os componentes do modelo base
# ------------------------------------------------------------

img2img_pipe = (
    StableDiffusionImg2ImgPipeline(
        vae=txt2img_pipe.vae,
        text_encoder=txt2img_pipe.text_encoder,
        tokenizer=txt2img_pipe.tokenizer,
        unet=txt2img_pipe.unet,
        scheduler=txt2img_pipe.scheduler,
        safety_checker=None,
        feature_extractor=(
            txt2img_pipe.feature_extractor
        ),
        image_encoder=getattr(
            txt2img_pipe,
            "image_encoder",
            None
        ),
        requires_safety_checker=False,
    )
)


img2img_pipe = img2img_pipe.to(
    DEVICE
)

img2img_pipe.enable_attention_slicing()


generator = torch.Generator(
    device=DEVICE
).manual_seed(
    COMPARISON_SEED
)


with torch.inference_mode():

    image_img2img = img2img_pipe(

        prompt=comparison_prompt,

        image=comparison_draft,

        strength=IMG2IMG_STRENGTH,

        num_inference_steps=(
            COMPARISON_STEPS
        ),

        guidance_scale=(
            COMPARISON_GUIDANCE
        ),

        generator=generator,

    ).images[0]


img2img_path = os.path.join(
    COMPARISON_DIR,
    f"{comparison_id}_02_img2img.png"
)

image_img2img.save(
    img2img_path
)

print(
    "Salvo:",
    img2img_path
)


# ============================================================
# Libera Stable Diffusion base
# ============================================================

del img2img_pipe
del txt2img_pipe

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# GLIGEN volta para GPU
# ------------------------------------------------------------

pipe.to(DEVICE)

print(
    "\nGLIGEN carregado novamente na GPU."
)


# ============================================================
# D. MÉTODO 3
# ZERO-SHOT-SLD SEM IMAGEM DE REFERÊNCIA
#
# Prompt
#   ↓
# LLM
#   ↓
# Bounding Boxes
#   ↓
# GLIGEN
# ============================================================

print("\n" + "=" * 70)
print(
    "3/4 — ZERO-SHOT-SLD "
    "SEM IMAGEM DE REFERÊNCIA"
)
print("=" * 70)


layout_prompt_only = (
    build_zero_shot_layout(

        user_prompt=(
            comparison_prompt
        ),

        sketch_image=None,
    )
)


prompt_boxes_generation_prompt = (
    layout_prompt_only[
        "generation_prompt"
    ]
)

prompt_boxes_phrases = (
    layout_prompt_only[
        "phrases"
    ]
)

prompt_boxes = (
    layout_prompt_only[
        "boxes"
    ]
)


print(
    "\nPrompt consolidado:"
)

print(
    prompt_boxes_generation_prompt
)


print(
    "\nBounding boxes:"
)

for phrase, box in zip(
    prompt_boxes_phrases,
    prompt_boxes
):
    print(
        phrase,
        "->",
        box
    )


image_boxes_prompt = (
    generate_from_layout(

        generation_prompt=(
            prompt_boxes_generation_prompt
        ),

        phrases=(
            prompt_boxes_phrases
        ),

        boxes=(
            prompt_boxes
        ),

        seed=(
            COMPARISON_SEED
        ),

        num_inference_steps=(
            COMPARISON_STEPS
        ),

        guidance_scale=(
            COMPARISON_GUIDANCE
        ),
    )
)


boxes_prompt_path = os.path.join(
    COMPARISON_DIR,
    f"{comparison_id}_03_boxes_prompt.png"
)

image_boxes_prompt.save(
    boxes_prompt_path
)


# ------------------------------------------------------------
# Cria visualização das boxes sem referência
# ------------------------------------------------------------

white_canvas = Image.new(
    "RGB",
    (
        IMAGE_SIZE,
        IMAGE_SIZE
    ),
    "white"
)


layout_prompt_only_vis = draw_layout(
    white_canvas,
    prompt_boxes_phrases,
    prompt_boxes
)


layout_prompt_only_vis.save(
    os.path.join(
        COMPARISON_DIR,
        f"{comparison_id}_03_layout.png"
    )
)


# ============================================================
# E. MÉTODO 4
# ZERO-SHOT-SLD COM IMAGEM DE REFERÊNCIA
#
# Prompt + Sketch
#       ↓
#      LLM
#       ↓
# Bounding Boxes
#       ↓
#     GLIGEN
#
# O SKETCH NÃO É ENVIADO PARA O GLIGEN
# ============================================================

print("\n" + "=" * 70)
print(
    "4/4 — ZERO-SHOT-SLD "
    "COM IMAGEM DE REFERÊNCIA"
)
print("=" * 70)


layout_with_reference = (
    build_zero_shot_layout(

        user_prompt=(
            comparison_prompt
        ),

        sketch_image=(
            comparison_draft
        ),
    )
)


reference_generation_prompt = (
    layout_with_reference[
        "generation_prompt"
    ]
)

reference_phrases = (
    layout_with_reference[
        "phrases"
    ]
)

reference_boxes = (
    layout_with_reference[
        "boxes"
    ]
)


print(
    "\nPrompt consolidado:"
)

print(
    reference_generation_prompt
)


print(
    "\nBounding boxes:"
)

for phrase, box in zip(
    reference_phrases,
    reference_boxes
):
    print(
        phrase,
        "->",
        box
    )


image_boxes_reference = (
    generate_from_layout(

        generation_prompt=(
            reference_generation_prompt
        ),

        phrases=(
            reference_phrases
        ),

        boxes=(
            reference_boxes
        ),

        seed=(
            COMPARISON_SEED
        ),

        num_inference_steps=(
            COMPARISON_STEPS
        ),

        guidance_scale=(
            COMPARISON_GUIDANCE
        ),
    )
)


boxes_reference_path = os.path.join(
    COMPARISON_DIR,
    f"{comparison_id}_04_boxes_reference.png"
)

image_boxes_reference.save(
    boxes_reference_path
)


# ------------------------------------------------------------
# Salva visualização das boxes sobre o sketch
# ------------------------------------------------------------

layout_reference_vis = draw_layout(

    comparison_draft,

    reference_phrases,

    reference_boxes
)


layout_reference_vis.save(

    os.path.join(

        COMPARISON_DIR,

        f"{comparison_id}_04_layout.png"
    )
)


# ============================================================
# F. VISUALIZAÇÃO COMPARATIVA
# ============================================================

print("\n" + "=" * 70)
print("RESULTADOS")
print("=" * 70)


fig, axes = plt.subplots(

    1,
    4,

    figsize=(22, 6)
)


# ------------------------------------------------------------
# 1. Prompt only
# ------------------------------------------------------------

axes[0].imshow(
    image_prompt_only
)

axes[0].set_title(
    "1. Somente Prompt",
    fontsize=15,
    fontweight="bold"
)

axes[0].axis(
    "off"
)


# ------------------------------------------------------------
# 2. Img2Img
# ------------------------------------------------------------

axes[1].imshow(
    image_img2img
)

axes[1].set_title(
    "2. Img2Img\nstrength = 0.7",
    fontsize=15,
    fontweight="bold"
)

axes[1].axis(
    "off"
)


# ------------------------------------------------------------
# 3. Boxes sem referência
# ------------------------------------------------------------

axes[2].imshow(
    image_boxes_prompt
)

axes[2].set_title(
    "3. Zero-shot-SLD\n"
    "LLM + Prompt",
    fontsize=15,
    fontweight="bold"
)

axes[2].axis(
    "off"
)


# ------------------------------------------------------------
# 4. Boxes com referência
# ------------------------------------------------------------

axes[3].imshow(
    image_boxes_reference
)

axes[3].set_title(
    "4. Zero-shot-SLD\n"
    "LLM + Prompt + Rascunho",
    fontsize=15,
    fontweight="bold"
)

axes[3].axis(
    "off"
)


# ------------------------------------------------------------
# Prompt abaixo da comparação
# ------------------------------------------------------------

wrapped_prompt = "\n".join(

    textwrap.wrap(

        comparison_prompt,

        width=120
    )
)


fig.suptitle(

    "Comparação dos Métodos de Geração",

    fontsize=20,
    fontweight="bold"
)


fig.text(

    0.5,
    0.02,

    "Prompt: "
    + wrapped_prompt,

    ha="center",
    fontsize=11
)


plt.tight_layout(

    rect=[
        0,
        0.07,
        1,
        0.93
    ]
)


# ============================================================
# G. SALVA COMPARAÇÃO
# ============================================================

comparison_path = os.path.join(

    COMPARISON_DIR,

    f"{comparison_id}_comparacao.png"
)


plt.savefig(

    comparison_path,

    dpi=200,

    bbox_inches="tight"
)


plt.show()


print("\nComparação salva em:")
print(comparison_path)

## Diferença conceitual para o SLD

O SLD original executa uma estratégia de **self-correction em loop fechado**:
primeiro existe uma geração, depois o resultado é inspecionado e corrigido.

O Zero-shot-SLD deste notebook executa **planejamento/condicionamento espacial
pré-geração**:

**intenção → layout explícito → primeira geração condicionada**

Assim, não há imagem inicial a ser corrigida.

Se existir um rascunho, ele é somente uma fonte adicional de intenção e de
informação espacial para a LLM. Nenhum pixel do rascunho entra no modelo de
difusão.

